# FEEDIT Dictionary DB Load

Django ORM을 직접 사용해서 Dictionary 데이터를 RDS에 적재하는 노트북.

- 각 Dictionary 종류별로 셀 분리
- CSV / XLSX / XLSM 지원
- 현재는 **Category 셀부터 실제 적재 가능**
- 나머지 Dictionary 셀은 같은 노트북에서 순차적으로 추가/실행


In [3]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

candidates = [
    PROJECT_ROOT / "backend",
    PROJECT_ROOT,
    PROJECT_ROOT.parent / "backend",
]

BACKEND_DIR = None
for p in candidates:
    if (p / "manage.py").exists():
        BACKEND_DIR = p.resolve()
        break

if BACKEND_DIR is None:
    raise FileNotFoundError("backend/manage.py 위치를 찾지 못했습니다.")

sys.path.insert(0, str(BACKEND_DIR))

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")

# Jupyter에서 sync ORM 허용
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

import django
django.setup()

print("BACKEND_DIR =", BACKEND_DIR)
print("Django setup 완료")

BACKEND_DIR = C:\SKN31-FINAL-4Team\backend
Django setup 완료


In [2]:
# 1. 공통 유틸: CSV / Excel 읽기
import csv
from pathlib import Path
from openpyxl import load_workbook


def clean(value):
    if value is None:
        return None
    if isinstance(value, str):
        value = value.strip()
        return value or None
    return value


def read_table(file_path, sheet_name=None, encoding=None):
    path = Path(file_path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        encodings = [encoding] if encoding else ["utf-8-sig", "utf-8", "cp949"]

        last_error = None
        for enc in encodings:
            try:
                with path.open("r", encoding=enc, newline="") as f:
                    reader = csv.DictReader(f)
                    rows = []
                    for row in reader:
                        item = {
                            str(k).strip(): clean(v)
                            for k, v in row.items()
                            if k is not None
                        }
                        if any(v is not None for v in item.values()):
                            rows.append(item)
                    return rows
            except UnicodeDecodeError as e:
                last_error = e

        raise last_error

    if suffix in {".xlsx", ".xlsm"}:
        wb = load_workbook(path, read_only=True, data_only=True)

        if sheet_name is None:
            sheet_name = wb.sheetnames[0]

        if sheet_name not in wb.sheetnames:
            raise ValueError(
                f"시트 없음: {sheet_name} / 사용 가능: {wb.sheetnames}"
            )

        ws = wb[sheet_name]
        values = ws.iter_rows(values_only=True)
        headers = [clean(v) for v in next(values)]

        rows = []
        for row in values:
            item = {
                headers[i]: clean(row[i])
                for i in range(min(len(headers), len(row)))
                if headers[i]
            }
            if any(v is not None for v in item.values()):
                rows.append(item)

        return rows

    raise ValueError("지원 형식: .csv, .xlsx, .xlsm")


def preview(rows, n=5):
    print("rows =", len(rows))
    for row in rows[:n]:
        print(row)


## CATEGORY

In [ ]:
# 2. CATEGORY 적재
from django.db import transaction
from backend.apps.core.models.dictionary import Category

CATEGORY_FILE = Path(
    r"C:\SKN31-FINAL-4Team\backend\temp_data\[DB_DICT]_CATEGORY.csv"
)

category_rows = read_table(CATEGORY_FILE)
preview(category_rows)

rows = 78
{'id': None, 'category_code': 'TOP', 'parent_category_code': None, 'name': '상의', 'level': '1', 'sort_order': '10', 'status': 'ACTIVE', 'created_at': None, 'updated_at': None}
{'id': None, 'category_code': 'OUTER', 'parent_category_code': None, 'name': '아우터', 'level': '1', 'sort_order': '20', 'status': 'ACTIVE', 'created_at': None, 'updated_at': None}
{'id': None, 'category_code': 'BOTTOM', 'parent_category_code': None, 'name': '하의', 'level': '1', 'sort_order': '30', 'status': 'ACTIVE', 'created_at': None, 'updated_at': None}
{'id': None, 'category_code': 'DRESS_SKIRT', 'parent_category_code': None, 'name': '원피스·스커트', 'level': '1', 'sort_order': '40', 'status': 'ACTIVE', 'created_at': None, 'updated_at': None}
{'id': None, 'category_code': 'SHOES', 'parent_category_code': None, 'name': '신발', 'level': '1', 'sort_order': '50', 'status': 'ACTIVE', 'created_at': None, 'updated_at': None}


In [9]:
preview_rows = []

for row in category_rows:
    code = clean(row.get("category_code") or row.get("code"))
    parent_code = clean(row.get("parent_category_code"))

    if not code:
        continue

    item = {
        "code": code,
        "name": clean(row.get("name")),
        "parent_code": parent_code,
        "level": int(row.get("level") or 1),
        "sort_order": (
            int(row["sort_order"])
            if clean(row.get("sort_order")) is not None
            else None
        ),
        "status": clean(row.get("status")) or "ACTIVE",
    }

    preview_rows.append(item)

print(f"적재 예정 건수: {len(preview_rows)}")
print("-" * 80)

for item in preview_rows:
    print(item)

적재 예정 건수: 78
--------------------------------------------------------------------------------
{'code': 'TOP', 'name': '상의', 'parent_code': None, 'level': 1, 'sort_order': 10, 'status': 'ACTIVE'}
{'code': 'OUTER', 'name': '아우터', 'parent_code': None, 'level': 1, 'sort_order': 20, 'status': 'ACTIVE'}
{'code': 'BOTTOM', 'name': '하의', 'parent_code': None, 'level': 1, 'sort_order': 30, 'status': 'ACTIVE'}
{'code': 'DRESS_SKIRT', 'name': '원피스·스커트', 'parent_code': None, 'level': 1, 'sort_order': 40, 'status': 'ACTIVE'}
{'code': 'SHOES', 'name': '신발', 'parent_code': None, 'level': 1, 'sort_order': 50, 'status': 'ACTIVE'}
{'code': 'ACCESSORY', 'name': '잡화·액세서리', 'parent_code': None, 'level': 1, 'sort_order': 60, 'status': 'ACTIVE'}
{'code': 'BAG', 'name': '가방', 'parent_code': None, 'level': 1, 'sort_order': 70, 'status': 'ACTIVE'}
{'code': 'ETC', 'name': '기타', 'parent_code': None, 'level': 1, 'sort_order': 80, 'status': 'ACTIVE'}
{'code': 'TOP_SHORT_SLEEVE_TSHIRT', 'name': '반소매 티셔츠', 'parent_cod

In [10]:
import os
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

In [11]:
created = 0
updated = 0
skipped = 0

with transaction.atomic():

    for row in category_rows:
        code = clean(row.get("category_code") or row.get("code"))

        if not code:
            skipped += 1
            continue

        obj, was_created = Category.objects.update_or_create(
            code=code,
            defaults={
                "name": clean(row.get("name")),
                "level": int(row.get("level") or 1),
                "sort_order": (
                    int(row["sort_order"])
                    if clean(row.get("sort_order")) is not None
                    else None
                ),
                "status": clean(row.get("status")) or Category.Status.ACTIVE,
            },
        )

        created += int(was_created)
        updated += int(not was_created)

    parent_linked = 0

    for row in category_rows:
        code = clean(row.get("category_code") or row.get("code"))
        parent_code = clean(row.get("parent_category_code"))

        if not code or not parent_code:
            continue

        obj = Category.objects.get(code=code)
        parent = Category.objects.get(code=parent_code)

        if obj.pk == parent.pk:
            raise ValueError(f"자기 자신을 parent로 지정할 수 없음: {code}")

        if obj.parent_id != parent.id:
            obj.parent = parent
            obj.save(update_fields=["parent"])
            parent_linked += 1

print({
    "created": created,
    "updated": updated,
    "parent_linked": parent_linked,
    "skipped": skipped,
})

{'created': 78, 'updated': 0, 'parent_linked': 70, 'skipped': 0}


In [12]:
# 4. CATEGORY 적재 확인
qs = (
    Category.objects
    .select_related("parent")
    .order_by("level", "sort_order", "code")
)

print("총 Category:", qs.count())

for c in qs[:30]:
    print(
        c.id,
        c.code,
        c.name,
        "parent=",
        c.parent.code if c.parent else None,
        "level=",
        c.level,
    )


총 Category: 78
1 TOP 상의 parent= None level= 1
2 OUTER 아우터 parent= None level= 1
3 BOTTOM 하의 parent= None level= 1
4 DRESS_SKIRT 원피스·스커트 parent= None level= 1
5 SHOES 신발 parent= None level= 1
6 ACCESSORY 잡화·액세서리 parent= None level= 1
7 BAG 가방 parent= None level= 1
8 ETC 기타 parent= None level= 1
67 ACC_HAT 모자 parent= ACCESSORY level= 2
66 BAG_ALL 가방 parent= BAG level= 2
39 BOTTOM_DENIM 데님 팬츠 parent= BOTTOM level= 2
48 DRESS_MINI 미니 원피스 parent= DRESS_SKIRT level= 2
78 ETC_INNERWEAR 이너웨어 parent= ETC level= 2
19 OUTER_WINDBREAKER 바람막이 parent= OUTER level= 2
57 SHOES_SNEAKERS 스니커즈 parent= SHOES level= 2
9 TOP_SHORT_SLEEVE_TSHIRT 반소매 티셔츠 parent= TOP level= 2
68 ACC_JEWELRY 주얼리 parent= ACCESSORY level= 2
40 BOTTOM_COTTON_CHINO 코튼/치노 팬츠 parent= BOTTOM level= 2
49 DRESS_MIDI 미디 원피스 parent= DRESS_SKIRT level= 2
20 OUTER_TRAINING_JACKET 트레이닝 재킷 parent= OUTER level= 2
58 SHOES_SPORTS 스포츠화 parent= SHOES level= 2
10 TOP_LONG_SLEEVE_TSHIRT 긴소매 티셔츠 parent= TOP level= 2
69 ACC_EYEWEAR 아이웨어 parent= ACCES

## DICTIONARY_TERM

In [3]:
import os
import sys
from pathlib import Path

BACKEND_DIR = Path(r"C:\SKN31-FINAL-4Team\backend")

sys.path.insert(0, str(BACKEND_DIR))

os.environ["DJANGO_SETTINGS_MODULE"] = "config.settings"
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

import django
django.setup()

In [6]:

created = 0
updated = 0
skipped = 0

STYLE_FILE = Path(
    r"C:\SKN31-FINAL-4Team\backend\temp_data\[DB_DICT]_STYLE.csv"
)
style_rows = read_table(STYLE_FILE)
preview(style_rows)

created = 0
updated = 0
skipped = 0

for row in style_rows:
    term_code = clean(row.get("term_code"))

    if not term_code:
        skipped += 1
        continue

    obj, was_created = DictionaryTerm.objects.update_or_create(
        term_code=term_code,
        defaults={
            "term_type": clean(row.get("term_type")) or "STYLE",
            "canonical_name": clean(row.get("canonical_name")),
            "normalized_name": clean(row.get("normalized_name"))
            or clean(row.get("canonical_name")),
            "english_name": clean(row.get("english_name")),
            "description": clean(row.get("description")),
            "status": clean(row.get("status")) or "ACTIVE",
        },
    )

    created += int(was_created)
    updated += int(not was_created)

print({
    "created": created,
    "updated": updated,
    "skipped": skipped,
})

rows = 14
{'term_id': '1', 'term_code': 'STYLE_BALLETCORE', 'term_type': 'STYLE', 'canonical_name': '발레코어', 'style_group': '코어(하위문화 믹스)', 'is_core': 'TRUE', 'created_at': None, 'updated_at': None}
{'term_id': '2', 'term_code': 'STYLE_GORPCORE', 'term_type': 'STYLE', 'canonical_name': '고프코어', 'style_group': '코어(하위문화 믹스)', 'is_core': 'TRUE', 'created_at': None, 'updated_at': None}
{'term_id': '3', 'term_code': 'STYLE_BLOKECORE', 'term_type': 'STYLE', 'canonical_name': '블록코어', 'style_group': '코어(하위문화 믹스)', 'is_core': 'TRUE', 'created_at': None, 'updated_at': None}
{'term_id': '4', 'term_code': 'STYLE_BIKERCORE', 'term_type': 'STYLE', 'canonical_name': '바이크코어', 'style_group': '코어(하위문화 믹스)', 'is_core': 'TRUE', 'created_at': None, 'updated_at': None}
{'term_id': '5', 'term_code': 'STYLE_GEEK_SHEEK', 'term_type': 'STYLE', 'canonical_name': '긱시크', 'style_group': '오피스/무드', 'is_core': 'TRUE', 'created_at': None, 'updated_at': None}
{'created': 14, 'updated': 0, 'skipped': 0}


## 스타일 적재


In [7]:
STYLE_FILE = Path(
    r"C:\SKN31-FINAL-4Team\backend\temp_data\[DB_DICT]_STYLE.csv"
)
style_rows = read_table(STYLE_FILE)
preview(style_rows)


created = 0
updated = 0
skipped = 0

for row in style_rows:
    term_code = clean(row.get("term_code"))

    if not term_code:
        skipped += 1
        continue

    term = DictionaryTerm.objects.get(
        term_code=term_code
    )

    raw_is_core = clean(row.get("is_core"))

    is_core = str(raw_is_core).strip().lower() in {
        "1",
        "true",
        "t",
        "y",
        "yes",
        "on",
    }

    obj, was_created = Style.objects.update_or_create(
        term=term,
        defaults={
            "style_group": clean(row.get("style_group")),
            "is_core": is_core,
        },
    )

    created += int(was_created)
    updated += int(not was_created)

print({
    "created": created,
    "updated": updated,
    "skipped": skipped,
})

for s in (
    Style.objects
    .select_related("term")
    .order_by("term__term_code")
):
    print(
        s.term_id,
        s.term.term_code,
        s.term.canonical_name,
        s.style_group,
        s.is_core,
    )


rows = 14
{'term_id': '1', 'term_code': 'STYLE_BALLETCORE', 'term_type': 'STYLE', 'canonical_name': '발레코어', 'style_group': '코어(하위문화 믹스)', 'is_core': 'TRUE', 'created_at': None, 'updated_at': None}
{'term_id': '2', 'term_code': 'STYLE_GORPCORE', 'term_type': 'STYLE', 'canonical_name': '고프코어', 'style_group': '코어(하위문화 믹스)', 'is_core': 'TRUE', 'created_at': None, 'updated_at': None}
{'term_id': '3', 'term_code': 'STYLE_BLOKECORE', 'term_type': 'STYLE', 'canonical_name': '블록코어', 'style_group': '코어(하위문화 믹스)', 'is_core': 'TRUE', 'created_at': None, 'updated_at': None}
{'term_id': '4', 'term_code': 'STYLE_BIKERCORE', 'term_type': 'STYLE', 'canonical_name': '바이크코어', 'style_group': '코어(하위문화 믹스)', 'is_core': 'TRUE', 'created_at': None, 'updated_at': None}
{'term_id': '5', 'term_code': 'STYLE_GEEK_SHEEK', 'term_type': 'STYLE', 'canonical_name': '긱시크', 'style_group': '오피스/무드', 'is_core': 'TRUE', 'created_at': None, 'updated_at': None}
{'created': 14, 'updated': 0, 'skipped': 0}
11 STYLE_AMEKAJI 아메카

## TERM_ALIAS

In [ ]:
# TERM_ALIAS 적재 셀
# 다음 단계에서 이 셀에 TERM_ALIAS 전용 FK/컬럼 매핑 로직을 넣으면 됨.
# CATEGORY와 독립적으로 실행 가능하도록 셀을 분리해 둠.

# 예:
# TERM_ALIAS_FILE = BACKEND_DIR / "data" / "dictionary" / "..."
# term_alias_rows = read_table(TERM_ALIAS_FILE, sheet_name="...")
# preview(term_alias_rows)

print("TERM_ALIAS - 아직 실행하지 않음")


## STYLE

In [ ]:
# STYLE 적재 셀
# 다음 단계에서 이 셀에 STYLE 전용 FK/컬럼 매핑 로직을 넣으면 됨.
# CATEGORY와 독립적으로 실행 가능하도록 셀을 분리해 둠.

# 예:
# STYLE_FILE = BACKEND_DIR / "data" / "dictionary" / "..."
# style_rows = read_table(STYLE_FILE, sheet_name="...")
# preview(style_rows)

print("STYLE - 아직 실행하지 않음")


## ITEM

In [ ]:
# ITEM 적재 셀
# 다음 단계에서 이 셀에 ITEM 전용 FK/컬럼 매핑 로직을 넣으면 됨.
# CATEGORY와 독립적으로 실행 가능하도록 셀을 분리해 둠.

# 예:
# ITEM_FILE = BACKEND_DIR / "data" / "dictionary" / "..."
# item_rows = read_table(ITEM_FILE, sheet_name="...")
# preview(item_rows)

print("ITEM - 아직 실행하지 않음")


## DETAIL

In [ ]:
# DETAIL 적재 셀
# 다음 단계에서 이 셀에 DETAIL 전용 FK/컬럼 매핑 로직을 넣으면 됨.
# CATEGORY와 독립적으로 실행 가능하도록 셀을 분리해 둠.

# 예:
# DETAIL_FILE = BACKEND_DIR / "data" / "dictionary" / "..."
# detail_rows = read_table(DETAIL_FILE, sheet_name="...")
# preview(detail_rows)

print("DETAIL - 아직 실행하지 않음")


## MATERIAL

In [ ]:
# MATERIAL 적재 셀
# 다음 단계에서 이 셀에 MATERIAL 전용 FK/컬럼 매핑 로직을 넣으면 됨.
# CATEGORY와 독립적으로 실행 가능하도록 셀을 분리해 둠.

# 예:
# MATERIAL_FILE = BACKEND_DIR / "data" / "dictionary" / "..."
# material_rows = read_table(MATERIAL_FILE, sheet_name="...")
# preview(material_rows)

print("MATERIAL - 아직 실행하지 않음")


## COLOR

In [ ]:
from backend.apps.core.models.dictionary import DictionaryTerm

FILE = Path(
    r"C:\SKN31-FINAL-4Team\backend\temp_data\[DB_DICT]_COLOR.csv"
)
color_rows = read_table(FILE)
preview(color_rows)


created = 0
updated = 0
skipped = 0

for row in color_rows:
    term_code = clean(row.get("term_code"))

    if not term_code:
        skipped += 1
        continue

    obj, was_created = DictionaryTerm.objects.update_or_create(
        term_code=term_code,
        defaults={
            "term_type": "COLOR",
            "canonical_name": clean(row.get("canonical_name")),
            "normalized_name": clean(row.get("canonical_name")),
            "english_name": clean(row.get("english_name")),
            "status": "ACTIVE",
        },
    )

    created += int(was_created)
    updated += int(not was_created)

print({
    "created": created,
    "updated": updated,
    "skipped": skipped,
})


rows = 82
{'term_id': None, 'term_code': 'COLOR_WHITE', 'term_type': 'COLOR', 'canonical_name': '화이트', 'english_name': 'White', 'color_family': '뉴트럴', 'base_color_id': None, 'base_color_code': None, 'base_color_name': None, 'note': '가장 기본적인 무채색, 미니멀·클린걸룩의 기본 컬러', 'created_at': None, 'updated_at': None}
{'term_id': None, 'term_code': 'COLOR_OFF_WHITE', 'term_type': 'COLOR', 'canonical_name': '오프화이트', 'english_name': 'Off-White', 'color_family': '뉴트럴', 'base_color_id': None, 'base_color_code': None, 'base_color_name': None, 'note': '순백보다 은은하게 아이보리 기가 도는 화이트', 'created_at': None, 'updated_at': None}
{'term_id': None, 'term_code': 'COLOR_IVORY', 'term_type': 'COLOR', 'canonical_name': '아이보리', 'english_name': 'Ivory', 'color_family': '뉴트럴', 'base_color_id': None, 'base_color_code': None, 'base_color_name': None, 'note': '부드럽고 따뜻한 인상의 크림빛 화이트', 'created_at': None, 'updated_at': None}
{'term_id': None, 'term_code': 'COLOR_CREAM', 'term_type': 'COLOR', 'canonical_name': '크림', 'english_name': '

In [ ]:
from backend.apps.core.models.dictionary import DictionaryTerm, Color

created = 0
updated = 0
skipped = 0

for row in color_rows:
    term_code = clean(row.get("term_code"))

    if not term_code:
        skipped += 1
        continue

    term = DictionaryTerm.objects.get(
        term_code=term_code
    )

    obj, was_created = Color.objects.update_or_create(
        term=term,
        defaults={
            "color_family": clean(row.get("color_family")),
            "note": clean(row.get("note")),
        },
    )

    created += int(was_created)
    updated += int(not was_created)

print({
    "created": created,
    "updated": updated,
    "skipped": skipped,
})

{'created': 82, 'updated': 0, 'skipped': 0}


## TPO

In [ ]:
from backend.apps.core.models.dictionary import DictionaryTerm
from backend.apps.core.models.dictionary import DictionaryTerm, TPO

FILE = Path(
    r"C:\SKN31-FINAL-4Team\backend\temp_data\[DB_DICT]_TPO.csv"
)
tpo_rows = read_table(FILE)

created = 0
updated = 0
skipped = 0

for row in tpo_rows:
    term_code = clean(row.get("term_code"))

    if not term_code:
        skipped += 1
        continue

    obj, was_created = DictionaryTerm.objects.update_or_create(
        term_code=term_code,
        defaults={
            "term_type": "TPO",
            "canonical_name": clean(row.get("canonical_name")),
            "normalized_name": clean(row.get("canonical_name")),
            "english_name": clean(row.get("english_name")),
            "status": "ACTIVE",
        },
    )

    created += int(was_created)
    updated += int(not was_created)

print({
    "created": created,
    "updated": updated,
    "skipped": skipped,
})

{'created': 55, 'updated': 0, 'skipped': 0}


In [ ]:
from backend.apps.core.models.dictionary import DictionaryTerm, TPO

FILE = Path(
    r"C:\SKN31-FINAL-4Team\backend\temp_data\[DB_DICT]_TPO.csv"
)
tpo_rows = read_table(FILE)
preview(tpo_rows)

created = 0
updated = 0
skipped = 0

for row in tpo_rows:
    term_code = clean(row.get("term_code"))

    if not term_code:
        skipped += 1
        continue

    term = DictionaryTerm.objects.get(
        term_code=term_code
    )

    obj, was_created = TPO.objects.update_or_create(
        term=term,
        defaults={
            "tpo_type": clean(row.get("tpo_type")),
            "note": clean(row.get("note")),
        },
    )

    created += int(was_created)
    updated += int(not was_created)

print({
    "created": created,
    "updated": updated,
    "skipped": skipped,
})



rows = 55
{'term_id': None, 'term_code': 'TPO_DAILY', 'term_type': 'TPO', 'canonical_name': '데일리', 'english_name': 'Daily', 'tpo_type': '시간', 'note': '특별한 목적 없이 일상적으로 착용하는 상황을 통칭', 'created_at': None, 'updated_at': None}
{'term_id': None, 'term_code': 'TPO_WEEKDAY', 'term_type': 'TPO', 'canonical_name': '평일', 'english_name': 'Weekday', 'tpo_type': '시간', 'note': '주말과 대비되는 일과 중심의 시간대', 'created_at': None, 'updated_at': None}
{'term_id': None, 'term_code': 'TPO_WEEKEND', 'term_type': 'TPO', 'canonical_name': '주말', 'english_name': 'Weekend', 'tpo_type': '시간', 'note': '평일과 대비되는 여유로운 시간대', 'created_at': None, 'updated_at': None}
{'term_id': None, 'term_code': 'TPO_MORNING', 'term_type': 'TPO', 'canonical_name': '아침', 'english_name': 'Morning', 'tpo_type': '시간', 'note': '등교·출근 등 하루를 시작하는 시간대', 'created_at': None, 'updated_at': None}
{'term_id': None, 'term_code': 'TPO_EVENING', 'term_type': 'TPO', 'canonical_name': '저녁', 'english_name': 'Evening', 'tpo_type': '시간', 'note': '퇴근 후, 저녁 약속 등에 어울리

## BRAND

In [3]:
BRAND_CATEGORY_MAP = {
    "국내 캐주얼·스트릿": "BRAND_DOMESTIC_CASUAL_STREET",
    "신발·스니커즈": "BRAND_SHOES_SNEAKERS",
    "국내 컨템포러리·디자이너": "BRAND_DOMESTIC_CONTEMPORARY_DESIGNER",
    "해외 럭셔리·컨템포러리": "BRAND_GLOBAL_LUXURY_CONTEMPORARY",
    "글로벌 스포츠·캐주얼": "BRAND_GLOBAL_SPORTS_CASUAL",
    "스포츠·액티브": "BRAND_SPORTS_ACTIVE",
    "아웃도어": "BRAND_OUTDOOR",
    "골프·스포츠라이프": "BRAND_GOLF_SPORTS_LIFE",
    "키즈": "BRAND_KIDS",
    "라이선스·콜라보": "BRAND_LICENSE_COLLAB",
    "국내 SPA·베이직": "BRAND_DOMESTIC_SPA",
    "무신사 자체 브랜드": "BRAND_MUSINSA",
}

print(len(BRAND_CATEGORY_MAP))

12


In [ ]:
from backend.apps.core.models.dictionary import Brand, Category

created = 0
updated = 0
skipped = 0

BRAND_FILE = Path(
    r"C:\SKN31-FINAL-4Team\backend\temp_data\[DB_DICT]_BRAND.csv"
)
brand_rows = read_table(BRAND_FILE)
preview(brand_rows)



for row in brand_rows:

    brand_code = clean(row.get("brand_code"))
    name = clean(row.get("canonical_name"))
    raw_category = clean(row.get("category"))

    if not brand_code or not name:
        skipped += 1
        continue

    category = None

    if raw_category:
        category_code = BRAND_CATEGORY_MAP.get(raw_category)

        if not category_code:
            print("⚠️ 알 수 없는 브랜드 카테고리:", raw_category)
            skipped += 1
            continue

        category = Category.objects.get(
            category_type="BRAND",
            code=category_code,
        )

    obj, was_created = Brand.objects.update_or_create(
        brand_code=brand_code,
        defaults={
            "name": name,
            "english_name": clean(row.get("english_name")),
            "country_code": clean(row.get("country_code")),
            "category": category,
            "description": clean(row.get("description")),
            "status": clean(row.get("status")) or "ACTIVE",
        },
    )

    created += int(was_created)
    updated += int(not was_created)

print({
    "created": created,
    "updated": updated,
    "skipped": skipped,
})

rows = 2775
{'id': None, 'brand_code': 'BRAND_GAKKAI_UNIONS', 'canonical_name': '가까이 유니언즈', 'normalized_name': '가까이 유니언즈', 'english_name': 'GAKKAI UNIONS', 'country_code': None, 'category': '국내 캐주얼·스트릿', 'description': None, 'status': 'ACTIVE', 'created_at': None, 'updated_at': None}
{'id': None, 'brand_code': 'BRAND_GAP', 'canonical_name': '갭', 'normalized_name': '갭', 'english_name': 'GAP', 'country_code': None, 'category': '글로벌 스포츠·캐주얼', 'description': '신발겸업', 'status': 'ACTIVE', 'created_at': None, 'updated_at': None}
{'id': None, 'brand_code': 'BRAND_GUESS', 'canonical_name': '게스', 'normalized_name': '게스', 'english_name': 'GUESS', 'country_code': None, 'category': '글로벌 스포츠·캐주얼', 'description': None, 'status': 'ACTIVE', 'created_at': None, 'updated_at': None}
{'id': None, 'brand_code': 'BRAND_KENZO', 'canonical_name': '겐조', 'normalized_name': '겐조', 'english_name': 'KENZO', 'country_code': None, 'category': '해외 럭셔리·컨템포러리', 'description': None, 'status': 'ACTIVE', 'created_at': None, 

## TERM_RELATION

In [ ]:
# TERM_RELATION 적재 셀
# 다음 단계에서 이 셀에 TERM_RELATION 전용 FK/컬럼 매핑 로직을 넣으면 됨.
# CATEGORY와 독립적으로 실행 가능하도록 셀을 분리해 둠.

# 예:
# TERM_RELATION_FILE = BACKEND_DIR / "data" / "dictionary" / "..."
# term_relation_rows = read_table(TERM_RELATION_FILE, sheet_name="...")
# preview(term_relation_rows)

print("TERM_RELATION - 아직 실행하지 않음")


In [6]:
from pathlib import Path

FILE = Path(
    r"C:\SKN31-FINAL-4Team\backend\temp_data\FEEDIT_ITEM_DETAIL_DB_FINAL.xlsx"
)

item_rows = read_table(
    FILE,
    sheet_name="ITEM_FINAL",
)

detail_rows = read_table(
    FILE,
    sheet_name="DETAIL_FINAL",
)

print("ITEM:", len(item_rows))
print("DETAIL:", len(detail_rows))

preview(item_rows)
preview(detail_rows)

ITEM: 74
DETAIL: 67
rows = 74
{'term_code': 'ITEM_SWEATSHIRT', 'term_type': 'ITEM', 'canonical_name': '맨투맨', 'english_name': 'Sweatshirt', 'category_code': 'TOP_SWEATSHIRT', 'gender_scope': 'ALL', 'note': '기모나 두께감 있는 저지·스웨트 원단으로 만든 대표적인 캐주얼 상의, 라운드넥이 기본형'}
{'term_code': 'ITEM_HOODIE', 'term_type': 'ITEM', 'canonical_name': '후드티', 'english_name': 'Hoodie', 'category_code': 'TOP_HOODIE', 'gender_scope': None, 'note': '모자(후드)가 달린 풀오버형 상의, 앞주머니(캥거루 포켓)가 있는 경우가 많음'}
{'term_code': 'ITEM_HOODED_ZIP_UP', 'term_type': 'ITEM', 'canonical_name': '후드집업', 'english_name': 'Hooded Zip-up', 'category_code': 'OUTER_HOOD_ZIPUP', 'gender_scope': None, 'note': '지퍼로 여미는 후드형 상의, 앞이 막힌 풀오버형 후드티와 구분됨'}
{'term_code': 'ITEM_T_SHIRT', 'term_type': 'ITEM', 'canonical_name': '티셔츠', 'english_name': 'T-shirt', 'category_code': 'TOP_SHORT_SLEEVE_TSHIRT', 'gender_scope': None, 'note': '가장 기본적인 반팔·긴팔 상의, 소재·핏·넥라인에 따라 다양하게 파생됨'}
{'term_code': 'ITEM_SHIRT', 'term_type': 'ITEM', 'canonical_name': '셔츠', 'english_name': 'Sh

In [ ]:
from backend.apps.core.models.dictionary import DictionaryTerm

created = 0
updated = 0
skipped = 0

for row in item_rows:

    term_code = clean(row.get("term_code"))

    if not term_code:
        skipped += 1
        continue

    obj, was_created = DictionaryTerm.objects.update_or_create(
        term_code=term_code,
        defaults={
            "term_type": "ITEM",
            "canonical_name": clean(row.get("canonical_name")),
            "normalized_name": clean(row.get("canonical_name")),
            "english_name": clean(row.get("english_name")),
            "status": "ACTIVE",
        },
    )

    created += int(was_created)
    updated += int(not was_created)

print({
    "created": created,
    "updated": updated,
    "skipped": skipped,
})

In [11]:
from pathlib import Path

FILE = Path(
    r"C:\SKN31-FINAL-4Team\backend\temp_data\FEEDIT_ITEM_DETAIL_DB_FINAL.xlsx"
)

item_rows = read_table(
    FILE,
    sheet_name="ITEM_FINAL",
)

detail_rows = read_table(
    FILE,
    sheet_name="DETAIL_FINAL",
)

print("ITEM:", len(item_rows))
print("DETAIL:", len(detail_rows))

preview(item_rows)
preview(detail_rows)

ITEM: 74
DETAIL: 67
rows = 74
{'term_code': 'ITEM_SWEATSHIRT', 'term_type': 'ITEM', 'canonical_name': '맨투맨', 'english_name': 'Sweatshirt', 'category_code': 'TOP_SWEATSHIRT', 'gender_scope': 'ALL', 'note': '기모나 두께감 있는 저지·스웨트 원단으로 만든 대표적인 캐주얼 상의, 라운드넥이 기본형'}
{'term_code': 'ITEM_HOODIE', 'term_type': 'ITEM', 'canonical_name': '후드티', 'english_name': 'Hoodie', 'category_code': 'TOP_HOODIE', 'gender_scope': None, 'note': '모자(후드)가 달린 풀오버형 상의, 앞주머니(캥거루 포켓)가 있는 경우가 많음'}
{'term_code': 'ITEM_HOODED_ZIP_UP', 'term_type': 'ITEM', 'canonical_name': '후드집업', 'english_name': 'Hooded Zip-up', 'category_code': 'OUTER_HOOD_ZIPUP', 'gender_scope': None, 'note': '지퍼로 여미는 후드형 상의, 앞이 막힌 풀오버형 후드티와 구분됨'}
{'term_code': 'ITEM_T_SHIRT', 'term_type': 'ITEM', 'canonical_name': '티셔츠', 'english_name': 'T-shirt', 'category_code': 'TOP_SHORT_SLEEVE_TSHIRT', 'gender_scope': None, 'note': '가장 기본적인 반팔·긴팔 상의, 소재·핏·넥라인에 따라 다양하게 파생됨'}
{'term_code': 'ITEM_SHIRT', 'term_type': 'ITEM', 'canonical_name': '셔츠', 'english_name': 'Sh

In [ ]:
from backend.apps.core.models.dictionary import DictionaryTerm

created = 0
updated = 0
skipped = 0

for row in item_rows:

    term_code = clean(row.get("term_code"))

    if not term_code:
        skipped += 1
        continue

    obj, was_created = DictionaryTerm.objects.update_or_create(
        term_code=term_code,
        defaults={
            "term_type": "ITEM",
            "canonical_name": clean(row.get("canonical_name")),
            "normalized_name": clean(row.get("canonical_name")),
            "english_name": clean(row.get("english_name")),
            "status": "ACTIVE",
        },
    )

    created += int(was_created)
    updated += int(not was_created)

print({
    "created": created,
    "updated": updated,
    "skipped": skipped,
})

{'created': 74, 'updated': 0, 'skipped': 0}


In [13]:
print(
    "ITEM DictionaryTerm:",
    DictionaryTerm.objects.filter(term_type="ITEM").count()
)

ITEM DictionaryTerm: 74


In [ ]:
from backend.apps.core.models.dictionary import (
    Category,
    DictionaryTerm,
    Item,
)

created = 0
updated = 0
skipped = 0

for row in item_rows:

    term_code = clean(row.get("term_code"))
    category_code = clean(row.get("category_code"))

    if not term_code:
        skipped += 1
        continue

    term = DictionaryTerm.objects.get(
        term_code=term_code
    )

    category = None

    if category_code:
        try:
            category = Category.objects.get(
                code=category_code,
                category_type="PRODUCT",
            )
        except Category.DoesNotExist:
            print(
                "❌ CATEGORY 없음:",
                term_code,
                "→",
                category_code,
            )
            skipped += 1
            continue

    obj, was_created = Item.objects.update_or_create(
        term=term,
        defaults={
            "category": category,
            "gender_scope": clean(row.get("gender_scope")),
            "note": clean(row.get("note")),
        },
    )

    created += int(was_created)
    updated += int(not was_created)

print({
    "created": created,
    "updated": updated,
    "skipped": skipped,
})

❌ CATEGORY 없음: ITEM_SETUP → ETC_SETUP
❌ CATEGORY 없음: ITEM_TRACK_TOP → ETC_SPORTSWEAR
❌ CATEGORY 없음: ITEM_RASH_GUARD → ETC_SPORTSWEAR
{'created': 71, 'updated': 0, 'skipped': 3}


In [15]:
qs = (
    Item.objects
    .select_related("term", "category")
    .order_by("term__term_code")
)

print("총 ITEM:", qs.count())

for item in qs[:30]:
    print(
        item.term.term_code,
        item.term.canonical_name,
        "→",
        item.category.code if item.category else None,
    )

총 ITEM: 71
ITEM_BAGGY_PANTS 배기팬츠 → BOTTOM_OTHER
ITEM_BALLET_FLATS 발레플랫슈즈 → SHOES_FLAT
ITEM_BERMUDA_SHORTS 버뮤다팬츠 → BOTTOM_SHORTS
ITEM_BIKER_JACKET 라이더자켓 → OUTER_LEATHER_JACKET
ITEM_BLAZER 블레이저 → OUTER_BLAZER_SUIT
ITEM_BLOUSE 블라우스 → TOP_BLOUSE
ITEM_BLOUSON 블루종 → OUTER_BLOUSON_MA1
ITEM_BOLERO 볼레로 → OUTER_OTHER
ITEM_BOOTS 부츠 → SHOES_BOOTS
ITEM_CANVAS_SHOES 캔버스화 → SHOES_SNEAKERS
ITEM_CAPRI_PANTS 카프리팬츠 → BOTTOM_OTHER
ITEM_CARDIGAN 가디건 → OUTER_CARDIGAN
ITEM_CARGO_PANTS 카고팬츠 → BOTTOM_CARGO
ITEM_CHELSEA_BOOTS 첼시부츠 → SHOES_BOOTS
ITEM_CHINO_PANTS 치노팬츠 → BOTTOM_COTTON_CHINO
ITEM_CLOG 클로그 → SHOES_SANDAL_SLIPPER
ITEM_COAT 코트 → OUTER_LONG_COAT
ITEM_COMBAT_BOOTS 워커 → SHOES_BOOTS
ITEM_CROP_TOP 크롭탑 → TOP_OTHER
ITEM_CULOTTES 큐롯팬츠 → DRESS_SKIRT_OTHER
ITEM_DENIM_JACKET 데님자켓 → OUTER_DENIM_JACKET
ITEM_DERBY_SHOES 더비슈즈 → SHOES_DERBY_LACEUP
ITEM_DOWN_JACKET 패딩 → OUTER_OTHER
ITEM_DRESS 원피스 → DRESS_MIDI
ITEM_DUFFLE_COAT 더플코트 → OUTER_SHORT_HALF_COAT
ITEM_FIELD_JACKET 야상 → OUTER_ANORAK
ITEM_FLATS 플랫슈즈 → SHOES_FLAT

In [16]:
created = 0
updated = 0
skipped = 0

for row in detail_rows:

    term_code = clean(row.get("term_code"))

    if not term_code:
        skipped += 1
        continue

    obj, was_created = DictionaryTerm.objects.update_or_create(
        term_code=term_code,
        defaults={
            "term_type": "DETAIL",
            "canonical_name": clean(row.get("canonical_name")),
            "normalized_name": clean(row.get("canonical_name")),
            "english_name": clean(row.get("english_name")),
            "status": "ACTIVE",
        },
    )

    created += int(was_created)
    updated += int(not was_created)

print({
    "created": created,
    "updated": updated,
    "skipped": skipped,
})

{'created': 67, 'updated': 0, 'skipped': 0}


In [17]:
print(
    "DETAIL DictionaryTerm:",
    DictionaryTerm.objects.filter(
        term_type="DETAIL"
    ).count()
)

DETAIL DictionaryTerm: 67


In [ ]:
from backend.apps.core.models.dictionary import Detail

created = 0
updated = 0
skipped = 0

for row in detail_rows:

    term_code = clean(row.get("term_code"))

    if not term_code:
        skipped += 1
        continue

    term = DictionaryTerm.objects.get(
        term_code=term_code
    )

    obj, was_created = Detail.objects.update_or_create(
        term=term,
        defaults={
            "target_type": clean(row.get("target_type")),
            "attribute_type": clean(row.get("attribute_type")),
            "note": clean(row.get("note")),
        },
    )

    created += int(was_created)
    updated += int(not was_created)

print({
    "created": created,
    "updated": updated,
    "skipped": skipped,
})

{'created': 67, 'updated': 0, 'skipped': 0}


In [4]:
from pathlib import Path

FILE = Path(
    r"C:\SKN31-FINAL-4Team\backend\temp_data\FEEDIT_ITEM_DETAIL_DB_FINAL_V2.xlsx"
)

item_rows = read_table(FILE, sheet_name="ITEM_FINAL")

print("ITEM_FINAL:", len(item_rows))

ITEM_FINAL: 98


In [ ]:
from backend.apps.core.models.dictionary import DictionaryTerm, Item, Category

new_item_rows = []

for row in item_rows:
    term_code = clean(row.get("term_code"))

    if not DictionaryTerm.objects.filter(term_code=term_code).exists():
        new_item_rows.append(row)

print("신규 ITEM:", len(new_item_rows))

for row in new_item_rows:
    print(
        row["term_code"],
        row["canonical_name"],
        "→",
        row["category_code"],
    )

신규 ITEM: 24
ITEM_LEG_WARMER 레그워머 → ACC_LEGWEAR
ITEM_UTILITY_BAG 유틸리티백 → BAG_ALL
ITEM_BEANIE 비니 → ACC_HAT
ITEM_CARABINER 카라비너 → ACC_OTHER
ITEM_SCARF 스카프 → ACC_SCARF_MUFFLER
ITEM_FOOTBALL_SOCKS 축구 양말 → ACC_LEGWEAR
ITEM_BELT 벨트 → ACC_BELT
ITEM_SUNGLASSES 선글라스 → ACC_EYEWEAR
ITEM_HORN_RIMMED_GLASSES 뿔테안경 → ACC_EYEWEAR
ITEM_BRIEFCASE 브리프케이스 → BAG_ALL
ITEM_BALL_CAP 볼캡 → ACC_HAT
ITEM_BACKPACK 백팩 → BAG_ALL
ITEM_NECKTIE 넥타이 → ACC_OTHER
ITEM_TOTE_BAG 토트백 → BAG_ALL
ITEM_SPORTS_SOCKS 스포츠 양말 → ACC_LEGWEAR
ITEM_CROSSBODY_BAG 크로스백 → BAG_ALL
ITEM_SPORTS_BRA 스포츠브라 → ETC_INNERWEAR
ITEM_KNIT_TIE 니트타이 → ACC_OTHER
ITEM_HAIRBAND 헤어밴드 → ACC_HAIR
ITEM_CANVAS_BAG 캔버스백 → BAG_ALL
ITEM_UTILITY_BELT 유틸리티벨트 → ACC_BELT
ITEM_JEWELRY 주얼리 → ACC_JEWELRY
ITEM_GLASSES 안경 → ACC_EYEWEAR
ITEM_BASKET_BAG 바스켓백 → BAG_ALL


In [6]:
created = 0
updated = 0

for row in new_item_rows:
    term_code = clean(row.get("term_code"))

    obj, was_created = DictionaryTerm.objects.update_or_create(
        term_code=term_code,
        defaults={
            "term_type": "ITEM",
            "canonical_name": clean(row.get("canonical_name")),
            "normalized_name": clean(row.get("canonical_name")),
            "english_name": clean(row.get("english_name")),
            "status": "ACTIVE",
        },
    )

    created += int(was_created)
    updated += int(not was_created)

print({
    "created": created,
    "updated": updated,
})

{'created': 24, 'updated': 0}


In [7]:
created = 0
updated = 0
skipped = 0

for row in new_item_rows:
    term_code = clean(row.get("term_code"))
    category_code = clean(row.get("category_code"))

    term = DictionaryTerm.objects.get(
        term_code=term_code
    )

    try:
        category = Category.objects.get(
            code=category_code,
            category_type="PRODUCT",
        )
    except Category.DoesNotExist:
        print("❌ CATEGORY 없음:", term_code, category_code)
        skipped += 1
        continue

    obj, was_created = Item.objects.update_or_create(
        term=term,
        defaults={
            "category": category,
            "gender_scope": clean(row.get("gender_scope")),
            "note": clean(row.get("note")),
        },
    )

    created += int(was_created)
    updated += int(not was_created)

print({
    "created": created,
    "updated": updated,
    "skipped": skipped,
})

{'created': 24, 'updated': 0, 'skipped': 0}


In [8]:
print(
    "ITEM TERM:",
    DictionaryTerm.objects.filter(term_type="ITEM").count()
)

print(
    "ITEM:",
    Item.objects.count()
)

ITEM TERM: 98
ITEM: 95


In [9]:
REL_FILE = Path(
    r"C:\SKN31-FINAL-4Team\backend\temp_data\FEEDIT_TERM_RELATION_FINAL_V2.xlsx"
)

relation_rows = read_table(
    REL_FILE,
    sheet_name="DB_READY",
)

print("TERM_RELATION:", len(relation_rows))
preview(relation_rows)

TERM_RELATION: 294
rows = 294
{'source_term_code': 'STYLE_BALLETCORE', 'target_term_code': 'ITEM_KNITWEAR', 'relation_type': 'RELATED_ITEM', 'weight': None, 'confidence': None, 'relation_source': 'MANUAL'}
{'source_term_code': 'STYLE_BALLETCORE', 'target_term_code': 'ITEM_CROP_TOP', 'relation_type': 'RELATED_ITEM', 'weight': None, 'confidence': None, 'relation_source': 'MANUAL'}
{'source_term_code': 'STYLE_BALLETCORE', 'target_term_code': 'ITEM_DRESS', 'relation_type': 'RELATED_ITEM', 'weight': None, 'confidence': None, 'relation_source': 'MANUAL'}
{'source_term_code': 'STYLE_BALLETCORE', 'target_term_code': 'ITEM_BALLET_FLATS', 'relation_type': 'RELATED_ITEM', 'weight': None, 'confidence': None, 'relation_source': 'MANUAL'}
{'source_term_code': 'STYLE_BALLETCORE', 'target_term_code': 'DETAIL_CROP_FIT', 'relation_type': 'RELATED_DETAIL', 'weight': None, 'confidence': None, 'relation_source': 'MANUAL'}


In [10]:
missing_source = []
missing_target = []

for row in relation_rows:
    source_code = clean(row.get("source_term_code"))
    target_code = clean(row.get("target_term_code"))

    if not DictionaryTerm.objects.filter(
        term_code=source_code
    ).exists():
        missing_source.append(source_code)

    if not DictionaryTerm.objects.filter(
        term_code=target_code
    ).exists():
        missing_target.append(target_code)

print("없는 SOURCE:", sorted(set(missing_source)))
print("없는 TARGET:", sorted(set(missing_target)))

없는 SOURCE: ['STYLE_CASUAL', 'STYLE_CHIC', 'STYLE_CITYBOY', 'STYLE_CLEAN_GIRL', 'STYLE_COTTAGECORE', 'STYLE_DARK_ACADEMIA', 'STYLE_GENDERLESS', 'STYLE_GIRLISH', 'STYLE_LOVELY', 'STYLE_MINIMAL', 'STYLE_PREPPY', 'STYLE_RETRO', 'STYLE_WORKWEAR']
없는 TARGET: ['MATERIAL_CANVAS', 'MATERIAL_CHIFFON', 'MATERIAL_CORDUROY', 'MATERIAL_COTTON', 'MATERIAL_DENIM', 'MATERIAL_DRY_DENIM', 'MATERIAL_FAUX_FUR', 'MATERIAL_FLEECE', 'MATERIAL_GABARDINE', 'MATERIAL_GORE_TEX', 'MATERIAL_HERRINGBONE', 'MATERIAL_JERSEY', 'MATERIAL_KNIT', 'MATERIAL_LACE', 'MATERIAL_LEATHER', 'MATERIAL_LINEN', 'MATERIAL_MESH', 'MATERIAL_NYLON', 'MATERIAL_POLYESTER', 'MATERIAL_RIPSTOP', 'MATERIAL_SATIN', 'MATERIAL_SELVAGE', 'MATERIAL_SILK', 'MATERIAL_SPANDEX', 'MATERIAL_STRETCH_DENIM', 'MATERIAL_TULLE', 'MATERIAL_TWEED', 'MATERIAL_TWILL', 'MATERIAL_VELVET', 'MATERIAL_WASHED', 'MATERIAL_WOOL']


In [11]:
missing_style_rows = [
    {
        "term_code": "STYLE_RETRO",
        "canonical_name": "레트로",
        "english_name": "Retro",
        "style_group": "레트로/무드",
        "is_core": False,
    },
    {
        "term_code": "STYLE_PREPPY",
        "canonical_name": "프레피",
        "english_name": "Preppy",
        "style_group": "무드/장르",
        "is_core": False,
    },
    {
        "term_code": "STYLE_GENDERLESS",
        "canonical_name": "젠더리스",
        "english_name": "Genderless",
        "style_group": "무드/장르",
        "is_core": False,
    },
    {
        "term_code": "STYLE_CITYBOY",
        "canonical_name": "시티보이",
        "english_name": "Cityboy",
        "style_group": "무드/장르",
        "is_core": False,
    },
    {
        "term_code": "STYLE_WORKWEAR",
        "canonical_name": "워크웨어",
        "english_name": "Workwear",
        "style_group": "무드/장르",
        "is_core": False,
    },
    {
        "term_code": "STYLE_MINIMAL",
        "canonical_name": "미니멀",
        "english_name": "Minimal",
        "style_group": "미니멀/무드",
        "is_core": False,
    },
    {
        "term_code": "STYLE_CASUAL",
        "canonical_name": "캐주얼",
        "english_name": "Casual",
        "style_group": "무드/장르",
        "is_core": False,
    },
    {
        "term_code": "STYLE_CHIC",
        "canonical_name": "시크",
        "english_name": "Chic",
        "style_group": "무드/장르",
        "is_core": False,
    },
    {
        "term_code": "STYLE_LOVELY",
        "canonical_name": "러블리",
        "english_name": "Lovely",
        "style_group": "무드/장르",
        "is_core": False,
    },
    {
        "term_code": "STYLE_GIRLISH",
        "canonical_name": "걸리시",
        "english_name": "Girlish",
        "style_group": "무드/장르",
        "is_core": False,
    },
    {
        "term_code": "STYLE_DARK_ACADEMIA",
        "canonical_name": "다크 아카데미아",
        "english_name": "Dark Academia",
        "style_group": "레트로/무드",
        "is_core": False,
    },
    {
        "term_code": "STYLE_COTTAGECORE",
        "canonical_name": "코티지코어",
        "english_name": "Cottagecore",
        "style_group": "코어(하위문화 믹스)",
        "is_core": False,
    },
    {
        "term_code": "STYLE_CLEAN_GIRL",
        "canonical_name": "클린 걸",
        "english_name": "Clean Girl",
        "style_group": "미니멀/무드",
        "is_core": False,
    },
]

In [ ]:
from backend.apps.core.models.dictionary import DictionaryTerm, Style

for row in missing_style_rows:
    DictionaryTerm.objects.update_or_create(
        term_code=row["term_code"],
        defaults={
            "term_type": "STYLE",
            "canonical_name": row["canonical_name"],
            "normalized_name": row["canonical_name"],
            "english_name": row["english_name"],
            "status": "ACTIVE",
        },
    )

print("STYLE TERM 추가 완료")

STYLE TERM 추가 완료


In [13]:
for row in missing_style_rows:
    term = DictionaryTerm.objects.get(
        term_code=row["term_code"]
    )

    Style.objects.update_or_create(
        term=term,
        defaults={
            "style_group": row["style_group"],
            "is_core": row["is_core"],
        },
    )

print("STYLE 본체 추가 완료")

STYLE 본체 추가 완료


In [14]:
for code in [
    "STYLE_RETRO",
    "STYLE_PREPPY",
    "STYLE_GENDERLESS",
    "STYLE_CITYBOY",
    "STYLE_WORKWEAR",
    "STYLE_MINIMAL",
    "STYLE_CASUAL",
    "STYLE_CHIC",
    "STYLE_LOVELY",
    "STYLE_GIRLISH",
    "STYLE_DARK_ACADEMIA",
    "STYLE_COTTAGECORE",
    "STYLE_CLEAN_GIRL",
]:
    print(
        code,
        DictionaryTerm.objects.filter(term_code=code).exists(),
        Style.objects.filter(term__term_code=code).exists(),
    )

STYLE_RETRO True True
STYLE_PREPPY True True
STYLE_GENDERLESS True True
STYLE_CITYBOY True True
STYLE_WORKWEAR True True
STYLE_MINIMAL True True
STYLE_CASUAL True True
STYLE_CHIC True True
STYLE_LOVELY True True
STYLE_GIRLISH True True
STYLE_DARK_ACADEMIA True True
STYLE_COTTAGECORE True True
STYLE_CLEAN_GIRL True True


In [16]:
from pathlib import Path

MATERIAL_FILE = Path(
    r"C:\SKN31-FINAL-4Team\backend\temp_data\FEEDIT_material_db_load.xlsx"
)

material_rows = read_table(
    MATERIAL_FILE,
    sheet_name="material_load",
)

print("MATERIAL rows:", len(material_rows))

for row in material_rows[:10]:
    print(
        row.get("term_code"),
        row.get("canonical_name"),
        row.get("material_type"),
        row.get("process_type"),
    )

MATERIAL rows: 65
MATERIAL_COTTON 코튼 천연섬유(식물성) 해당없음(조직 용어)
MATERIAL_LINEN 린넨 천연섬유(식물성) 해당없음(조직 용어)
MATERIAL_HEMP 헴프 천연섬유(식물성) 해당없음(조직 용어)
MATERIAL_RAMIE 저마 천연섬유(식물성) 해당없음(조직 용어)
MATERIAL_WOOL 울 천연섬유(동물성) 해당없음(조직 용어)
MATERIAL_CASHMERE 캐시미어 천연섬유(동물성) 해당없음(조직 용어)
MATERIAL_SILK 실크 천연섬유(동물성) 해당없음(조직 용어)
MATERIAL_ALPACA 알파카 천연섬유(동물성) 해당없음(조직 용어)
MATERIAL_MOHAIR 모헤어 천연섬유(동물성) 해당없음(조직 용어)
MATERIAL_ANGORA 앙고라 천연섬유(동물성) 해당없음(조직 용어)


In [ ]:
from backend.apps.core.models.dictionary import DictionaryTerm, Material

created = 0
updated = 0
skipped = 0

for row in material_rows:
    term_code = clean(row.get("term_code"))

    if not term_code:
        skipped += 1
        continue

    obj, was_created = DictionaryTerm.objects.update_or_create(
        term_code=term_code,
        defaults={
            "term_type": "MATERIAL",
            "canonical_name": clean(row.get("canonical_name")),
            "normalized_name": clean(row.get("canonical_name")),
            "english_name": clean(row.get("english_name")),
            "status": "ACTIVE",
        },
    )

    created += int(was_created)
    updated += int(not was_created)

print({
    "created": created,
    "updated": updated,
    "skipped": skipped,
})

{'created': 65, 'updated': 0, 'skipped': 0}


In [18]:
created = 0
updated = 0
skipped = 0

for row in material_rows:
    term_code = clean(row.get("term_code"))

    if not term_code:
        skipped += 1
        continue

    term = DictionaryTerm.objects.get(
        term_code=term_code
    )

    obj, was_created = Material.objects.update_or_create(
        term=term,
        defaults={
            "material_type": clean(row.get("material_type")),
            "process_type": clean(row.get("process_type")),
            "note": clean(row.get("note")),
        },
    )

    created += int(was_created)
    updated += int(not was_created)

print({
    "created": created,
    "updated": updated,
    "skipped": skipped,
})

{'created': 65, 'updated': 0, 'skipped': 0}


In [19]:
print(
    "MATERIAL TERM:",
    DictionaryTerm.objects.filter(
        term_type="MATERIAL"
    ).count()
)

print(
    "MATERIAL:",
    Material.objects.count()
)

MATERIAL TERM: 65
MATERIAL: 65


In [20]:
missing_material_codes = []

for row in relation_rows:
    target_code = clean(row.get("target_term_code"))

    if (
        target_code
        and target_code.startswith("MATERIAL_")
        and not DictionaryTerm.objects.filter(
            term_code=target_code
        ).exists()
    ):
        missing_material_codes.append(target_code)

print(
    "TERM_RELATION MATERIAL 누락:",
    sorted(set(missing_material_codes)),
)

TERM_RELATION MATERIAL 누락: []


In [21]:
missing_source = []
missing_target = []

for row in relation_rows:
    source_code = clean(row.get("source_term_code"))
    target_code = clean(row.get("target_term_code"))

    if not DictionaryTerm.objects.filter(term_code=source_code).exists():
        missing_source.append(source_code)

    if not DictionaryTerm.objects.filter(term_code=target_code).exists():
        missing_target.append(target_code)

print("없는 SOURCE:", sorted(set(missing_source)))
print("없는 TARGET:", sorted(set(missing_target)))

없는 SOURCE: []
없는 TARGET: []


In [22]:
missing_source = []
missing_target = []

for row in relation_rows:
    source_code = clean(row.get("source_term_code"))
    target_code = clean(row.get("target_term_code"))

    if not DictionaryTerm.objects.filter(term_code=source_code).exists():
        missing_source.append(source_code)

    if not DictionaryTerm.objects.filter(term_code=target_code).exists():
        missing_target.append(target_code)

print("없는 SOURCE:", sorted(set(missing_source)))
print("없는 TARGET:", sorted(set(missing_target)))

없는 SOURCE: []
없는 TARGET: []


In [ ]:
from django.db import transaction
from backend.apps.core.models.dictionary import DictionaryTerm, TermRelation


def clean_number(value):
    value = clean(value)

    if value in (None, ""):
        return None

    return value


created = 0
updated = 0
skipped = 0


# 매 행마다 DB 조회하지 않도록 미리 DictionaryTerm 캐싱
term_map = {
    term.term_code: term
    for term in DictionaryTerm.objects.all()
}


with transaction.atomic():

    for row in relation_rows:
        source_code = clean(row.get("source_term_code"))
        target_code = clean(row.get("target_term_code"))
        relation_type = clean(row.get("relation_type"))

        if not source_code or not target_code or not relation_type:
            print(
                "⚠️ SKIP:",
                source_code,
                target_code,
                relation_type,
            )
            skipped += 1
            continue

        source_term = term_map.get(source_code)
        target_term = term_map.get(target_code)

        if source_term is None:
            raise ValueError(
                f"SOURCE DictionaryTerm 없음: {source_code}"
            )

        if target_term is None:
            raise ValueError(
                f"TARGET DictionaryTerm 없음: {target_code}"
            )

        obj, was_created = TermRelation.objects.update_or_create(
            source_term=source_term,
            target_term=target_term,
            relation_type=relation_type,
            defaults={
                "weight": clean_number(row.get("weight")),
                "confidence": clean_number(row.get("confidence")),
                "relation_source": (
                    clean(row.get("relation_source"))
                    or "MANUAL"
                ),
            },
        )

        created += int(was_created)
        updated += int(not was_created)


print({
    "created": created,
    "updated": updated,
    "skipped": skipped,
})

{'created': 294, 'updated': 0, 'skipped': 0}


In [ ]:
print(
    "TERM_RELATION 전체:",
    TermRelation.objects.count()
)